# Swarm Pattern — Parallel Multi-Agent Execution

This notebook demonstrates:
- The Swarm multi-agent pattern (parallel execution)
- Coordinator agent that decomposes tasks and dispatches to workers
- Comparing sequential vs parallel agent execution
- Error handling when one agent in the swarm fails

## ⚠️ Cost Warning
- Multiple model invocations (3-4 agents per run)
- Estimated cost for this lab: **< $0.50** (Nova Pro is cost-effective)
- No persistent resources created

In [1]:
# Install required packages
!pip install strands-agents strands-agents-tools boto3 -q

In [2]:
import time
import concurrent.futures
from strands import Agent, tool
from strands.models import BedrockModel

REGION = "us-west-2"
MODEL_ID = "us.amazon.nova-pro-v1:0"

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
print(f"Model: {MODEL_ID} in {REGION}")

Model: us.amazon.nova-pro-v1:0 in us-west-2


## 1. Define Specialized Worker Agents

Each worker agent is an expert in one domain. The swarm dispatches sub-tasks to the right specialist.

In [3]:
# Specialist agents
security_agent = Agent(
    model=model,
    system_prompt="""You are an AWS security specialist. Provide concise security 
    recommendations for cloud architectures. Focus on IAM, encryption, and network security.
    Keep responses under 150 words."""
)

cost_agent = Agent(
    model=model,
    system_prompt="""You are an AWS cost optimization specialist. Provide concise cost 
    reduction recommendations. Focus on right-sizing, reserved capacity, and waste elimination.
    Keep responses under 150 words."""
)

performance_agent = Agent(
    model=model,
    system_prompt="""You are an AWS performance specialist. Provide concise performance 
    optimization recommendations. Focus on caching, scaling, and latency reduction.
    Keep responses under 150 words."""
)

print("Worker agents created: security, cost, performance")

Worker agents created: security, cost, performance


## 2. Sequential Execution (Baseline)

First, let's run all three agents sequentially to establish a time baseline.

In [4]:
TASK = "Review a serverless API using Lambda, API Gateway, and DynamoDB"

# Sequential execution
start_time = time.time()

security_result = security_agent(f"Provide security recommendations for: {TASK}")
cost_result = cost_agent(f"Provide cost optimization recommendations for: {TASK}")
performance_result = performance_agent(f"Provide performance recommendations for: {TASK}")

sequential_time = time.time() - start_time

print(f"=== Sequential Execution: {sequential_time:.2f}s ===")
print(f"\n🔒 Security:\n{str(security_result.message)[:200]}...")
print(f"\n💰 Cost:\n{str(cost_result.message)[:200]}...")
print(f"\n⚡ Performance:\n{str(performance_result.message)[:200]}...")

Ensure IAM roles for Lambda have least privilege, enabling only necessary permissions. Encrypt data at rest in DynamoDB using KMS. Enable encryption in transit for API Gateway with TLS. Use VPC endpoints for secure communication between API Gateway and DynamoDB. Implement rate limiting and throttling in API Gateway to prevent abuse. Regularly review and rotate IAM credentials. Enable AWS WAF for API Gateway to protect against common web exploits. Monitor with CloudTrail and set alarms in CloudWatch for unusual activity.Optimize Lambda memory allocation for cost efficiency. Use provisioned concurrency for predictable traffic. Utilize DynamoDB auto-scaling and consider on-demand capacity mode. Implement API Gateway usage plans to control costs. Analyze access patterns and consider using DAX for DynamoDB acceleration. Regularly review and delete unused resources.Optimize Lambda performance by using provisioned concurrency for predictable workloads. Implement caching at the API Gateway lev

## 3. Swarm Execution (Parallel)

Now let's run the same agents in parallel using the swarm pattern. All workers execute simultaneously.

In [5]:
def run_agent(agent, prompt):
    """Execute an agent and return its result."""
    result = agent(prompt)
    return str(result.message)

# Parallel execution using ThreadPoolExecutor
start_time = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    futures = {
        executor.submit(run_agent, security_agent, f"Provide security recommendations for: {TASK}"): "security",
        executor.submit(run_agent, cost_agent, f"Provide cost optimization recommendations for: {TASK}"): "cost",
        executor.submit(run_agent, performance_agent, f"Provide performance recommendations for: {TASK}"): "performance",
    }
    
    swarm_results = {}
    for future in concurrent.futures.as_completed(futures):
        agent_name = futures[future]
        try:
            swarm_results[agent_name] = future.result()
        except Exception as e:
            swarm_results[agent_name] = f"ERROR: {e}"

parallel_time = time.time() - start_time

print(f"=== Swarm (Parallel) Execution: {parallel_time:.2f}s ===")
print(f"\n🔒 Security:\n{str(swarm_results['security'])[:200]}...")
print(f"\n💰 Cost:\n{str(swarm_results['cost'])[:200]}...")
print(f"\n⚡ Performance:\n{str(swarm_results['performance'])[:200]}...")

Apply IAM roles withOptimize Lambda functionOptimize Lambda with memory and timeout settings. Use provisioned concurrency for predictable traffic. Implement API Gateway caching. Use DynamoDB auto-scaling and consider reserved capacity for consistent workloads. Analyze and delete provisioned concurrency and warm functions to reduce cold starts. Enable API Gateway caching for GET requests. Use DynamoDB DAX for frequent reads. Scale Lambda with auto-scaling based on metrics. Place API Gateway, least privilege for Lambda functions. Use KMS to encrypt DynamoDB data at rest. Enforce encryption in transit via API Gateway with TLS. Utilize VPC endpoints for secure API Gateway to DynamoDB communication. Implement WAF on API Gateway to unused DynamoDB tables and Lambda functions. Enable cost allocation tags for better tracking. block common threats. Enable CloudTrail logging and set up CloudWatch alarms for unusual activity. Regularly rotate IAM credentials and review permissions. Apply rate lim

## 4. Performance Comparison

In [6]:
speedup = sequential_time / parallel_time if parallel_time > 0 else 0

print(f"{'Approach':<20} {'Time (s)':<15} {'Speedup':<10}")
print("=" * 45)
print(f"{'Sequential':<20} {sequential_time:<15.2f} {'1.0x':<10}")
print(f"{'Swarm (Parallel)':<20} {parallel_time:<15.2f} {f'{speedup:.1f}x':<10}")
print(f"\n✅ Swarm pattern achieved {speedup:.1f}x speedup with 3 parallel agents")

Approach             Time (s)        Speedup   
Sequential           3.93            1.0x      
Swarm (Parallel)     1.35            2.9x      

✅ Swarm pattern achieved 2.9x speedup with 3 parallel agents


## 5. Coordinator Agent (Full Swarm Pattern)

A complete swarm has a coordinator that:
1. Decomposes the task into sub-tasks
2. Dispatches to worker agents in parallel
3. Synthesizes results into a unified response

In [7]:
@tool
def analyze_security(architecture_description: str) -> str:
    """Run security analysis on an architecture using a specialist agent.
    
    Args:
        architecture_description: Description of the architecture to analyze
    """
    result = security_agent(f"Analyze security for: {architecture_description}")
    return str(result.message)

@tool
def analyze_cost(architecture_description: str) -> str:
    """Run cost analysis on an architecture using a specialist agent.
    
    Args:
        architecture_description: Description of the architecture to analyze
    """
    result = cost_agent(f"Analyze costs for: {architecture_description}")
    return str(result.message)

@tool
def analyze_performance(architecture_description: str) -> str:
    """Run performance analysis on an architecture using a specialist agent.
    
    Args:
        architecture_description: Description of the architecture to analyze
    """
    result = performance_agent(f"Analyze performance for: {architecture_description}")
    return str(result.message)

# Coordinator agent
coordinator = Agent(
    model=model,
    tools=[analyze_security, analyze_cost, analyze_performance],
    system_prompt="""You are an AWS Well-Architected Review coordinator. When asked to review 
    an architecture, use ALL THREE analysis tools (security, cost, performance) to get specialist 
    opinions, then synthesize them into a unified recommendation report."""
)

print("Coordinator agent created with 3 specialist tools")

Coordinator agent created with 3 specialist tools


In [8]:
# Run the full swarm via coordinator
start_time = time.time()

review = coordinator(
    "Review this architecture: A serverless e-commerce API using Lambda, API Gateway, "
    "DynamoDB, and S3 for static assets. Expected traffic: 10,000 requests/minute peak."
)

coordinator_time = time.time() - start_time
print(f"\n=== Coordinator Review ({coordinator_time:.2f}s) ===")
print(review.message)

<thinking> To review the given architecture, I need to analyze it from three different perspectives: security, cost, and performance. Each analysis will provide insights that will help in forming a comprehensive recommendation report. I will use the provided tools to get specialist opinions on each aspect. </thinking>

Tool #1: analyze_security

Tool #2: analyze_cost

Tool #3: analyze_performance
For theImplement IAM roles with leastRight-size Lambda memory for cost efficiency. Use provisioned concurrency for predictable peaks. Enable API Gateway caching. Optimize DynamoDB read/write capacity with auto-scaling. Use S3 Intelligent-Tiering for static assets. Implement reserved e-commerce API, use provisioned concurrency for Lambda to handle peak traffic. Enable API Gateway caching and use DynamoDB DAX for read-heavy operations. Store static assets in S3 with CloudFront for global distribution. Implement auto-scaling for  capacity for consistent DynamoDB usage. Regularly review and delete

## 6. Error Handling in Swarms

What happens when one agent in the swarm fails? The pattern should handle partial failures gracefully.

In [9]:
def run_agent_with_timeout(agent, prompt, timeout=30):
    """Execute agent with timeout and error handling."""
    try:
        result = agent(prompt)
        return {"status": "success", "result": str(result.message)}
    except Exception as e:
        return {"status": "error", "error": str(e)}

# Simulate a swarm where one agent might fail
agents_config = [
    ("security", security_agent, f"Analyze: {TASK}"),
    ("cost", cost_agent, f"Analyze: {TASK}"),
    ("performance", performance_agent, f"Analyze: {TASK}"),
]

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    futures = {}
    for name, agent, prompt in agents_config:
        futures[executor.submit(run_agent_with_timeout, agent, prompt)] = name
    
    results = {}
    for future in concurrent.futures.as_completed(futures):
        name = futures[future]
        results[name] = future.result()

# Report results
print("Swarm Results:")
for name, result in results.items():
    status_icon = "✅" if result["status"] == "success" else "❌"
    print(f"  {status_icon} {name}: {result['status']}")

# Synthesize only successful results
successful = {k: v["result"] for k, v in results.items() if v["status"] == "success"}
failed = [k for k, v in results.items() if v["status"] == "error"]

if failed:
    print(f"\n⚠️ Partial failure: {failed} agents failed. Proceeding with {len(successful)} results.")
else:
    print(f"\n✅ All {len(successful)} agents completed successfully.")

For theEnsure IAM roles forOptimize Lambda memory Lambda have least privilege. Encrypt DynamoDB data at rest with KMS. Enforce encryption in transit via API Gateway using TLS. Utilize VPC endpoints for secure API Gateway to DynamoDB communication. Implement WAF on API Gateway to protect against common threats. Enable CloudTrail logging and set CloudWatch alarms for unusual and timeout settings. Use provisioned concurrency for consistent traffic. Implement API Gateway caching. Use DynamoDB auto-scaling and consider on-demand capacity mode. Analyze and delete unused resources. Enable cost allocation tags for better tracking. serverless API, optimize Lambda with provisioned concurrency and warm functions. Enable API Gateway caching for GET requests. Use DynamoDB DAX for caching frequent reads. Scale Lambda with auto-scaling based on concurrent executions. Place all services in the same region to minimize latency. Implement throttling and request activity. Regularly review and rotate IAM c

## Pattern Selection Guide

| Pattern | When to Use | Trade-offs |
|---------|-------------|------------|
| **Agents as Tools** | One agent delegates to another | Simple, sequential, easy to debug |
| **Swarm** | Independent sub-tasks in parallel | Fast, but no inter-agent communication |
| **Agent Graph** | Complex workflows with branching | Flexible, but more complex to set up |
| **Sequential Pipeline** | Each step depends on the previous | Predictable, but slower |

**Use Swarms when:**
- Sub-tasks are independent (no data dependencies between workers)
- Latency matters (parallel = faster)
- You can tolerate partial failures
- Workers don't need to communicate with each other

## 🧹 Cleanup (Optional)

No persistent resources created. All agents run in-memory.